# Content-Based Filtering

This notebook implements content-based recommendation systems using **The Movies Dataset** (Kaggle - Rounak Banik):
- Genre-based recommendations
- Evaluate content-based filtering performance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import models and evaluation from src
import sys
sys.path.append('..')
from src.models import GenreBasedRecommender, TFIDFContentRecommender
from src.evaluation import MetricsCalculator
from src.data import DataLoader

print("Modules loaded successfully!")

Modules loaded successfully!


# Import data preprocessor
from src.data.data_preprocessor import DataPreprocessor
preprocessor = DataPreprocessor()

# Load data with user-based split to ensure users appear in both train and test
train_ratings, test_ratings = preprocessor.create_train_test_split(ratings, test_size=0.2, method='user_based')

# Filter for overlapping users between train and test
train_users = set(train_ratings['userId'].unique())
test_users = set(test_ratings['userId'].unique())
overlap_users = train_users & test_users
print(f"Train users: {len(train_users)}, Test users: {len(test_users)}, Overlap: {len(overlap_users)}")

# Keep only overlapping users
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)]
test_ratings = test_ratings[test_ratings['userId'].isin(overlap_users)]

# Sample for faster training
train_ratings = train_ratings.sample(n=100000, random_state=42)
test_ratings = test_ratings.sample(n=20000, random_state=42)

print(f"Train ratings: {len(train_ratings)}")
print(f"Test ratings: {len(test_ratings)}")
print(f"Unique users in train: {train_ratings['userId'].nunique()}")
print(f"Unique movies in train: {train_ratings['movieId'].nunique()}")

In [2]:
# FIX #1 — Load from correctly split preprocessed files
from src.data.data_preprocessor import DataPreprocessor
loader = DataLoader()
movies_metadata, train_ratings, test_ratings = loader.load_preprocessed_data()

# Enforce overlap
overlap_users = set(train_ratings['userId']) & set(test_ratings['userId'])
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)].copy()
test_ratings  = test_ratings[test_ratings['userId'].isin(overlap_users)].copy()

print(f"Overlap users:  {len(overlap_users):,}")
print(f"Train ratings:  {len(train_ratings):,}")
print(f"Test ratings:   {len(test_ratings):,}")
print(f"Train movies:   {train_ratings['movieId'].nunique():,}")

assert len(overlap_users) >= 95, f"Too few overlap users: {len(overlap_users)}"
print("✓ Verification gate: overlap users ≥ 95")

Overlap users:  256,107
Train ratings:  20,881,168
Test ratings:   5,109,682
Train movies:   43,372
✓ Verification gate: overlap users ≥ 95


## Genre-Based Content Filtering

In [3]:
# Genre-based recommender (uses full train_ratings)
genre_recommender = GenreBasedRecommender(movies_metadata, train_ratings, use_keywords=True)

# FIX #3 — TF-IDF: max_features=10000, soup now includes genres 3× + director + top-3 cast
# The soup construction is inside _build_text_corpus in src/models/content_based.py
tfidf_recommender = TFIDFContentRecommender(
    movies_metadata,
    train_ratings,
    max_features=10000,
    min_df=2,
    max_df=0.8,
    min_user_rating=4.0,
)

# Sanity check
test_user = test_ratings['userId'].iloc[0]
print(f"\nSample genre recs for user {test_user}:")
print(genre_recommender.recommend_for_user(test_user, n=5)[['movieId', 'title', 'score']].to_string(index=False))
print(f"\nSample TF-IDF recs for user {test_user}:")
tfidf_recs = tfidf_recommender.recommend(test_user, n=5)
if not tfidf_recs.empty:
    print(tfidf_recs[['movieId', 'title', 'score']].to_string(index=False))

Genre-based model: 20 genres, 19387 keywords, 256107 users


TF-IDF content model: 42210 movies built

Sample genre recs for user 1:
 movieId                       title    score
     933            To Catch a Thief 6.884822
    1653                     Gattaca 6.757829
   71033    The Secret in Their Eyes 6.755285
    1150 The Return of Martin Guerre 6.633912
  103235              The Best Offer 6.482809

Sample TF-IDF recs for user 1:


 movieId                                    title    score
   40815      Harry Potter and the Goblet of Fire 0.341140
    8368 Harry Potter and the Prisoner of Azkaban 0.334050
   80933                             Murder, Inc. 0.332972
    8773    Sherlock Holmes and the Secret Weapon 0.332196
    2414                    Young Sherlock Holmes 0.327259


## Evaluate Content-Based Model

In [4]:
# Evaluate — identical settings across ALL notebooks: K=10, threshold=4.0, 100 users, seed=42
calculator = MetricsCalculator()
eval_kwargs = dict(k=10, n_users=100, min_rating=4.0, random_state=42,
                   train_ratings=train_ratings)

print("Evaluating Genre Content-Based Filtering...")
genre_metrics = calculator.evaluate_model_with_recommendations(
    genre_recommender, test_ratings, **eval_kwargs)

print("Evaluating TF-IDF Content-Based Filtering...")
tfidf_metrics = calculator.evaluate_model_with_recommendations(
    tfidf_recommender, test_ratings, **eval_kwargs)

for name, m in [("Genre Content-Based", genre_metrics),
                ("TF-IDF Content-Based", tfidf_metrics)]:
    print(f"\n{name}:")
    for k, v in m.items():
        print(f"  {k}: {v:.4f}")

# VERIFICATION GATE #3: TF-IDF precision must be > genre precision
# (confirms that improved soup is working)
print(f"\nGenre P@10:  {genre_metrics['precision@k']:.4f}")
print(f"TF-IDF P@10: {tfidf_metrics['precision@k']:.4f}")

# Save metrics
content_based_metrics = {'genre_based': genre_metrics, 'tfidf_based': tfidf_metrics}
calculator.save_metrics(content_based_metrics, '../reports/content_based_metrics.json')
print("\n✓ Content-based metrics saved → reports/content_based_metrics.json")

Evaluating Genre Content-Based Filtering...


Evaluating TF-IDF Content-Based Filtering...



Genre Content-Based:
  precision@k: 0.0107
  recall@k: 0.0066
  f1@k: 0.0082
  hit_rate: 0.0090
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0178

TF-IDF Content-Based:
  precision@k: 0.0071
  recall@k: 0.0073
  f1@k: 0.0072
  hit_rate: 0.0060
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0197

Genre P@10:  0.0107
TF-IDF P@10: 0.0071

✓ Content-based metrics saved → reports/content_based_metrics.json


## Save Content-Based Model

In [5]:
# Save metrics using MetricsCalculator
content_based_metrics = {
    'genre_based': genre_metrics,
    'tfidf_based': tfidf_metrics
}
calculator.save_metrics(content_based_metrics, '../reports/content_based_metrics.json')

print("Content-Based metrics saved successfully!")
print("\nSaved file:")
print("- reports/content_based_metrics.json")

Content-Based metrics saved successfully!

Saved file:
- reports/content_based_metrics.json


## Summary

In [6]:
print("Content-Based Filtering Summary:")
print(f"\nGenre Content-Based:")
for metric, value in genre_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nTF-IDF Content-Based:")
for metric, value in tfidf_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nBest performing model based on F1 score:")
best_model = max(['genre_based', 'tfidf_based'], 
                key=lambda x: content_based_metrics[x]['f1@k'])
print(f"  {best_model}: {content_based_metrics[best_model]['f1@k']:.4f}")

Content-Based Filtering Summary:

Genre Content-Based:
  precision@k: 0.0107
  recall@k: 0.0066
  f1@k: 0.0082
  hit_rate: 0.0090
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0178

TF-IDF Content-Based:
  precision@k: 0.0071
  recall@k: 0.0073
  f1@k: 0.0072
  hit_rate: 0.0060
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0197

Best performing model based on F1 score:
  genre_based: 0.0082
